# Week 4: Building WaveNet
This notebook implements a hierarchical (WaveNet-like) architecture to process a larger context window of characters (8) progressively.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# Download the dataset
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

In [ ]:
# Read in all the words
words = open('names.txt', 'r').read().splitlines()
print(f"Total words: {len(words)}")
print(words[:8])

In [ ]:
# Build the vocabulary
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(f"Vocab size: {vocab_size}")

In [ ]:
# Build the dataset with block_size = 8
block_size = 8 

def build_dataset(words):
  X, Y = [], []
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]
  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])     # 80%
Xdev, Ydev = build_dataset(words[n1:n2])   # 10%
Xte,  Yte  = build_dataset(words[n2:])     # 10%

In [ ]:
# Define the FlattenConsecutive Layer
# This is the core of the WaveNet architecture: grouping consecutive elements.

class FlattenConsecutive(torch.nn.Module):
    def __init__(self, n):
        super().__init__()
        self.n = n 
    
    def forward(self, x):
        B, T, C = x.shape
        # Group 'n' consecutive elements
        # (B, T, C) -> (B, T//n, C*n)
        x = x.view(B, T // self.n, C * self.n)
        
        # If we've crunched it down to a single element in the Time dimension,
        # squeeze it out to match the expected input for the final Linear layer.
        if x.shape[1] == 1:
            x = x.squeeze(1)
        return x

# Helper BatchNorm1d that handles (B, T, C) inputs correctly
class BatchNorm1d(torch.nn.BatchNorm1d):
    def forward(self, x):
        # PyTorch BatchNorm1d expects (N, C) or (N, C, L)
        # We have (N, L, C). We need to transpose to (N, C, L), apply BN, then transpose back.
        if x.dim() == 2:
            return super().forward(x)
        
        # (B, T, C) -> (B, C, T)
        x = x.permute(0, 2, 1)
        x = super().forward(x)
        # (B, C, T) -> (B, T, C)
        x = x.permute(0, 2, 1)
        return x

In [ ]:
torch.manual_seed(42)

n_embd = 24  # Embedding dimension
n_hidden = 128 # Hidden layer size

# The WaveNet Architecture
# Context: 8 chars
# Layer 1: groups 2 chars -> 4 vectors
# Layer 2: groups 2 vectors -> 2 vectors
# Layer 3: groups 2 vectors -> 1 vector

model = torch.nn.Sequential(
    torch.nn.Embedding(vocab_size, n_embd),
    
    # --- Layer 1 ---
    FlattenConsecutive(2), 
    torch.nn.Linear(n_embd * 2, n_hidden, bias=False), 
    BatchNorm1d(n_hidden), 
    torch.nn.Tanh(),
    
    # --- Layer 2 ---
    FlattenConsecutive(2), 
    torch.nn.Linear(n_hidden * 2, n_hidden, bias=False), 
    BatchNorm1d(n_hidden), 
    torch.nn.Tanh(),
    
    # --- Layer 3 ---
    FlattenConsecutive(2), 
    torch.nn.Linear(n_hidden * 2, n_hidden, bias=False), 
    BatchNorm1d(n_hidden), 
    torch.nn.Tanh(),
    
    # --- Output ---
    torch.nn.Linear(n_hidden, vocab_size),
)

# Parameter initialization tweak for stability
with torch.no_grad():
  model[-1].weight *= 0.1 

parameters = [p for p in model.parameters()]
print(f"Total Parameters: {sum(p.nelement() for p in parameters)}")
for p in parameters:
  p.requires_grad = True

In [ ]:
# Training
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):
  
  # Minibatch
  ix = torch.randint(0, Xtr.shape[0], (batch_size,))
  Xb, Yb = Xtr[ix], Ytr[ix] 
  
  # Forward pass
  logits = model(Xb)
  loss = F.cross_entropy(logits, Yb)
  
  # Backward pass
  for p in parameters:
    p.grad = None
  loss.backward()
  
  # Update
  lr = 0.1 if i < 150000 else 0.01 # Learning rate decay
  for p in parameters:
    p.data += -lr * p.grad

  # Track stats
  if i % 10000 == 0:
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())
  
  # Break early if needed, but 200k steps ensures good convergence
  # if i >= 1000: break

In [ ]:
plt.plot(torch.tensor(lossi).view(-1, 1000).mean(1))

In [ ]:
# Evaluate on Train and Val sets
# Goal: Validation loss < 2.2

@torch.no_grad()
def split_loss(split):
  x,y = {
    'train': (Xtr, Ytr),
    'val': (Xdev, Ydev),
    'test': (Xte, Yte),
  }[split]
  logits = model(x)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

In [ ]:
# Sample from the model
for _ in range(20):
    out = []
    context = [0] * block_size 
    while True:
      logits = model(torch.tensor([context]))
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    print(''.join(itos[i] for i in out))